# Trabalho 1 — Aquisição de Dados
## Onda de Calor Recorde no Centro-Oeste (agosto/2024) × Indicadores Econômicos (IBGE)

**Disciplina:** Ciência de Dados — Instituto de Computação (UFAM)
(Aquisição de Dados)

**Data da coleta:** gerada automaticamente — ver coluna `data_hora_coleta_utc` em `registro_proveniencia.csv` (Seção 8)

---

### Fontes utilizadas

| # | Fonte | Método | O que fornece |
|---|-------|--------|----------------|
| 1 | **Boletins/notícias sobre a onda de calor de agosto/2024** (INMET e INPE via imprensa) | **Web scraping** (`requests` + `BeautifulSoup`) | Temperatura máxima em Cuiabá e no Brasil, umidade mínima do ar, dias consecutivos acima de 40°C e focos de calor em Mato Grosso, por data de boletim |
| 2 | **API do IBGE** (Localidades + SIDRA/Agregados) | **API REST** | Lista dos municípios do Centro-Oeste (código e nome); **IPCA mensal** e **taxa de desocupação** (PNAD Contínua) |

**Chave de integração:** `ano_mes` (`AAAA-MM`). A granularidade diária dos boletins é convertida
para mensal antes do merge (detalhes na Seção 6).

### Decisões metodológicas importantes (leia antes de rodar)

- **IPCA e o Centro-Oeste.** Diferente do Amazonas, **as três capitais do Centro-Oeste (Brasília,
  Goiânia e Campo Grande) são áreas de coleta do IPCA** — Brasília desde a lista histórica original,
  Goiânia e Campo Grande desde a expansão de jan/2020 (tabela 7060). O notebook procura Brasília
  primeiro (por ser a mais antiga na série) e cai para Goiânia, depois Campo Grande, depois a Região
  Centro-Oeste e por fim o Brasil, só se necessário. A localidade realmente usada fica registrada na
  coluna `ipca_localidade` da base.
- **Períodos explícitos.** Os períodos são calculados a partir de `JANELA_INICIO`/`JANELA_FIM`
  (Seção 1). **Não** usamos o atalho relativo `-N` da API, que devolve "os últimos N períodos" na
  data de execução e deixaria a coleta fora da janela da onda de calor.
- **Desocupação trimestral.** A PNAD Contínua trimestral tem códigos `AAAA01`–`AAAA04`
  (trimestres), que **não** são meses. O código converte cada trimestre nos seus 3 meses
  (o valor do trimestre é repetido nos meses) e guarda o período original em
  `desocupacao_periodo_original`.
- **Valores manuais têm prioridade sobre os automáticos** (o regex sobre texto jornalístico pode
  errar), e cada valor tem uma coluna `origem_*` indicando de onde veio. Divergências entre
  automático e manual são listadas na Seção 6.1.
- **Rótulo `em_crise` por temperatura, não por município.** Diferente de desastres como enchente ou
  estiagem (medidos por nº de municípios afetados), uma onda de calor é melhor sinalizada pela
  temperatura em si: aqui `em_crise` usa `temp_max_cuiaba_c >= LIMIAR_CRISE_TEMP_C` (Seção 1).

### Observação sobre robustez

Páginas de notícias mudam de estrutura com o tempo. As células de scraping são defensivas
(try/except, regex tolerante, validação de faixa de valores), **gravam o HTML bruto (bytes originais)
antes de qualquer extração** e registram o hash SHA-256 no log de proveniência. Se algum site
retornar erro (403, timeout), é necessario ajuste a URL ou use a conferência manual (Seção 3).


## 1. Setup do ambiente

In [ ]:
# Bibliotecas — no Google Colab, requests/bs4/pandas/lxml já vêm instaladas.
import os
import re
import json
import time
import hashlib
import zipfile
import datetime as dt
import urllib.robotparser as robotparser
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    from unidecode import unidecode
except ImportError:
    %pip install -q unidecode
    from unidecode import unidecode

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Ambiente pronto.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.8 MB/s eta 0:00:00
Ambiente pronto.


In [ ]:
# ======================= CONFIGURAÇÃO (edite aqui) =======================
CONTATO_EMAIL = " "
USER_AGENT_NOME = "TesteAcademico"

# Janela temporal da análise (inclusive). Cobre antes/durante/depois da onda de calor de ago/2024.
JANELA_INICIO = "2023-09"
JANELA_FIM = "2024-12"

# Regra (ARBITRÁRIA, documentada) para o rótulo "em_crise": >= X°C em Cuiabá (o epicentro da onda).
LIMIAR_CRISE_TEMP_C = 40.0

DELAY_ENTRE_REQUISICOES = 2            # segundos
ROBOTS_INDETERMINADO_PROSSEGUIR = False  # se robots.txt não puder ser lido: True = raspa mesmo assim

# IDs de tabelas do SIDRA (conferidos por nome/metadados na Seção 4; troque aqui se necessário)
ID_TABELA_IPCA = 7060           # IPCA — variação mensal etc. (a partir de jan/2020)
ID_TABELA_DESOCUPACAO = 4099    # PNAD Contínua trimestral — taxa de desocupação (14 anos ou mais)

# Colab tem disco efêmero: para não perder tudo ao desconectar, monte o Drive.
MONTAR_DRIVE = False
INCLUIR_HTML_BRUTO_NO_ZIP = False   # HTML de matérias é conteúdo protegido: por padrão NÃO vai no zip

# ==========================================================================
BASE_DIR = "."
if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/trabalho1_ondacalor"

DIR_BRUTOS = os.path.join(BASE_DIR, "dados_brutos")
DIR_BRUTOS_SCRAPING = os.path.join(DIR_BRUTOS, "scraping_ondacalor")
DIR_BRUTOS_IBGE = os.path.join(DIR_BRUTOS, "api_ibge")
DIR_TRATADOS = os.path.join(BASE_DIR, "dados_tratados")

for d in [DIR_BRUTOS, DIR_BRUTOS_SCRAPING, DIR_BRUTOS_IBGE, DIR_TRATADOS]:
    os.makedirs(d, exist_ok=True)

HEADERS = {
    "User-Agent": (
        f"Mozilla/5.0 (compatible; {USER_AGENT_NOME}/1.1; "
        f"uso academico, sem fins comerciais; contato: {CONTATO_EMAIL})"
    )
}
if "SEU_EMAIL_AQUI" in CONTATO_EMAIL:
    print("[aviso] Preencha CONTATO_EMAIL: o User-Agent deve permitir que os sites contatem os autores.")

MESES_JANELA = list(pd.period_range(JANELA_INICIO, JANELA_FIM, freq="M").strftime("%Y-%m"))
print("Pastas:", DIR_BRUTOS, "|", DIR_TRATADOS)
print(f"Janela de análise: {MESES_JANELA[0]} a {MESES_JANELA[-1]} ({len(MESES_JANELA)} meses)")


Pastas: ./dados_brutos | ./dados_tratados
Janela de análise: 2023-09 a 2024-12 (16 meses)


In [ ]:
# --- Utilitários gerais: proveniência, hash, HTTP com retry, números pt-BR ----------
provenance_log = []


def agora_utc():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")


def sha256_bytes(conteudo: bytes) -> str:
    return hashlib.sha256(conteudo).hexdigest()


def registrar_provenancia(fonte, url, metodo, parametros=None, observacao="", arquivo=None, sha256=None):
    '''Anota fonte, URL exata, data/hora, método, parâmetros e (se houver) arquivo bruto + hash.'''
    registro = {
        "fonte": fonte,
        "url": url,
        "data_hora_coleta_utc": agora_utc(),
        "metodo": metodo,
        "parametros": json.dumps(parametros or {}, ensure_ascii=False),
        "observacao": observacao,
        "arquivo_bruto": arquivo,
        "sha256": sha256,
    }
    provenance_log.append(registro)
    return registro


def salvar_bruto(caminho, conteudo: bytes) -> str:
    '''Grava os bytes EXATOS recebidos e devolve o SHA-256.'''
    with open(caminho, "wb") as f:
        f.write(conteudo)
    return sha256_bytes(conteudo)


def salvar_json_bruto(caminho, obj) -> str:
    return salvar_bruto(caminho, json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))


def http_get(url, tentativas=3, timeout=30, espera=2, **kwargs):
    '''GET com retry/backoff para erros de rede e HTTP 429/5xx. Pode devolver resposta 4xx.'''
    ultimo_erro = None
    for i in range(1, tentativas + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, **kwargs)
            if resp.status_code in (429, 500, 502, 503, 504) and i < tentativas:
                time.sleep(espera * i)
                continue
            return resp
        except requests.exceptions.RequestException as e:
            ultimo_erro = e
            if i < tentativas:
                time.sleep(espera * i)
    raise ultimo_erro


def parse_num_ptbr(txt):
    '''Converte número em texto para float, tratando pt-BR ('1.234,5') e ponto decimal ('12.89').
    - '12,89' -> 12.89 | '12.89' -> 12.89 | '1.289' -> 1289 (milhar) | '1.234,5' -> 1234.5'''
    s = str(txt).strip()
    if not s:
        return None
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    elif "." in s and re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    try:
        return float(s)
    except ValueError:
        return None


print("Utilitários prontos.")


Utilitários prontos.


## 2. Verificação de `robots.txt`

Antes de raspar, checamos o `robots.txt` **de cada URL que será acessada** (não só da raiz do
domínio), usando nosso próprio User-Agent. Resultado possível:

- `permitido` — pode raspar;
- `bloqueado` — a URL é **pulada** pelo laço de scraping;
- `indeterminado` — não foi possível ler o `robots.txt` (erro de rede, 401/403, 5xx). Por
  padrão (postura conservadora) a URL também é **pulada**

Se o `robots.txt` declarar `Crawl-delay`, o maior entre ele e `DELAY_ENTRE_REQUISICOES` é usado.
Cada leitura de `robots.txt` entra no log de proveniência.


In [ ]:
_robots_cache = {}


def _carregar_robots(url_alvo):
    p = urlparse(url_alvo)
    base = f"{p.scheme}://{p.netloc}"
    if base in _robots_cache:
        return _robots_cache[base]

    robots_url = f"{base}/robots.txt"
    info = {"robots_url": robots_url, "parser": None, "status": "indeterminado", "detalhe": ""}
    try:
        resp = http_get(robots_url, tentativas=2, timeout=15)
        if resp.status_code == 200:
            rp = robotparser.RobotFileParser()
            rp.parse(resp.text.splitlines())
            info.update(parser=rp, status="ok", detalhe="HTTP 200")
        elif resp.status_code in (404, 410):
            info.update(status="sem_robots", detalhe=f"HTTP {resp.status_code}: sem robots.txt")
        else:  # 401/403/429/5xx...: não dá para saber a política do site
            info.update(status="indeterminado", detalhe=f"HTTP {resp.status_code}")
    except Exception as e:
        info["detalhe"] = f"erro de rede: {e}"

    registrar_provenancia(
        fonte=f"robots.txt de {p.netloc}", url=robots_url,
        metodo="GET (requests + urllib.robotparser)", parametros={"user_agent": USER_AGENT_NOME},
        observacao=f"status={info['status']}; {info['detalhe']}",
    )
    _robots_cache[base] = info
    return info


def checar_robots(url_alvo):
    '''Retorna dict com 'pode' ('permitido'|'bloqueado'|'indeterminado'), 'crawl_delay' e detalhes.'''
    info = _carregar_robots(url_alvo)
    delay = None
    if info["status"] == "ok":
        rp = info["parser"]
        pode = "permitido" if rp.can_fetch(USER_AGENT_NOME, url_alvo) else "bloqueado"
        delay = rp.crawl_delay(USER_AGENT_NOME)
    elif info["status"] == "sem_robots":
        pode = "permitido"
    else:
        pode = "indeterminado"
    return {"pode": pode, "crawl_delay": delay, "robots_url": info["robots_url"], "detalhe": info["detalhe"]}


print("Checagem de robots.txt pronta.")


Checagem de robots.txt pronta.


## 3. Fonte 1 — Web Scraping: boletins da onda de calor de agosto/2024 no Centro-Oeste

Entre meados de agosto e o início de setembro de 2024, o Centro-Oeste (com epicentro em Mato Grosso
e Mato Grosso do Sul) viveu uma sequência de recordes de temperatura, associada a umidade
relativa extremamente baixa e ao pior agosto em focos de incêndio desde 2010 (Inpe). O INMET não
publica um boletim único e estruturado para isso, mas os números (temperatura máxima em Cuiabá e no
Brasil, umidade mínima, dias consecutivos acima de 40°C) e os dados do Inpe (BDQueimadas) foram
republicados quase diariamente por portais de notícia, que é o que raspamos aqui.

> **Limitação conhecida:** os boletins de temperatura (Climatempo, SBT News) e os de queimadas
> (Poder360) vêm de fontes e dias diferentes; a série resultante mistura granularidades (algumas
> datas só têm temperatura, outras só têm foco de incêndio). Ver Seção 11 (próximos passos).

**Confira cada URL manualmente** (título, data, teor) antes de usar; ajuste/substitua livremente por
fontes equivalentes, de preferência **primárias** (INMET/BDMEP, Inpe/BDQueimadas, Cemaden).


In [ ]:
# Lista curada de páginas com boletins/números da onda de calor de agosto/2024.
# data_ref = data de referência atribuída pelo grupo (será comparada com a data publicada na página).
URLS_ONDACALOR = [
    {"data_ref": "2024-08-15", "fonte": "Climatempo",
     "url": "https://www.climatempo.com.br/noticia/2024/08/16/com-41dc-cuiaba-registra-novo-recorde-de-calor-6038"},
    {"data_ref": "2024-08-19", "fonte": "IHU/ClimaInfo",
     "url": "https://ihu.unisinos.br/642521-clima-extremo-nova-onda-de-calor-no-brasil-pode-trazer-recordes-de-temperatura-em-agosto"},
    {"data_ref": "2024-08-21", "fonte": "Climatempo",
     "url": "https://www.climatempo.com.br/noticia/2024/08/22/cuiaba-sete-dias-consecutivos-com-mais-de-40dc-6108"},
    {"data_ref": "2024-08-29", "fonte": "Climatempo",
     "url": "https://www.climatempo.com.br/noticia/2024/08/29/agosto-termina-com-mais-de-40dc-no-brasil-6213"},
    {"data_ref": "2024-08-30", "fonte": "Poder360",
     "url": "https://www.poder360.com.br/brasil/brasil-tem-agosto-com-maior-numero-de-queimadas-desde-2010/"},
    {"data_ref": "2024-09-03", "fonte": "Climatempo",
     "url": "https://www.climatempo.com.br/noticia/2024/09/04/calor-acima-dos-40dc-e-umidade-abaixo-de-10-no-brasil-6269"},
    {"data_ref": "2024-09-07", "fonte": "SBT News",
     "url": "https://sbtnews.sbt.com.br/noticia/brasil/cuiaba-registra-42-6-c-maior-temperatura-do-pais-segundo-inmet"},
    {"data_ref": "2024-09-10", "fonte": "Poder360",
     "url": "https://www.poder360.com.br/poder-sustentavel/brasil-registra-5-132-focos-de-incendio/"},
    {"data_ref": "2024-09-11", "fonte": "Poder360",
     "url": "https://www.poder360.com.br/poder-sustentavel/brasil-registra-2-909-focos-de-incendio/"},
    {"data_ref": "2024-09-24", "fonte": "Poder360",
     "url": "https://www.poder360.com.br/poder-sustentavel/brasil-registra-1-338-focos-de-incendio-sendo-57-na-amazonia/"},
]

df_urls = pd.DataFrame(URLS_ONDACALOR)
df_urls


,data_ref,fonte,url
0,2024-08-15,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...
1,2024-08-19,IHU/ClimaInfo,https://ihu.unisinos.br/642521-clima-extremo-n...
2,2024-08-21,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...
3,2024-08-29,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...
4,2024-08-30,Poder360,https://www.poder360.com.br/brasil/brasil-tem-...
5,2024-09-03,Climatempo,https://www.climatempo.com.br/noticia/2024/09/...
6,2024-09-07,SBT News,https://sbtnews.sbt.com.br/noticia/brasil/cuia...
7,2024-09-10,Poder360,https://www.poder360.com.br/poder-sustentavel/...
8,2024-09-11,Poder360,https://www.poder360.com.br/poder-sustentavel/...
9,2024-09-24,Poder360,https://www.poder360.com.br/poder-sustentavel/...


In [ ]:
# Checagem de robots.txt de CADA URL (com o nosso User-Agent). O resultado é usado pelo laço de scraping.
robots_resultado = []
for item in URLS_ONDACALOR:
    r = checar_robots(item["url"])
    item["robots"] = r["pode"]
    item["crawl_delay"] = r["crawl_delay"]
    robots_resultado.append({
        "dominio": urlparse(item["url"]).netloc, "url": item["url"], "robots_url": r["robots_url"],
        "resultado": r["pode"], "crawl_delay": r["crawl_delay"], "detalhe": r["detalhe"],
    })

df_robots = pd.DataFrame(robots_resultado)
print(df_robots["resultado"].value_counts().to_string())
df_robots


resultado
permitido    10


,dominio,url,robots_url,resultado,crawl_delay,detalhe
0,www.climatempo.com.br,https://www.climatempo.com.br/noticia/2024/08/...,https://www.climatempo.com.br/robots.txt,permitido,None,HTTP 200
1,ihu.unisinos.br,https://ihu.unisinos.br/642521-clima-extremo-n...,https://ihu.unisinos.br/robots.txt,permitido,None,HTTP 200
2,www.climatempo.com.br,https://www.climatempo.com.br/noticia/2024/08/...,https://www.climatempo.com.br/robots.txt,permitido,None,HTTP 200
3,www.climatempo.com.br,https://www.climatempo.com.br/noticia/2024/08/...,https://www.climatempo.com.br/robots.txt,permitido,None,HTTP 200
4,www.poder360.com.br,https://www.poder360.com.br/brasil/brasil-tem-...,https://www.poder360.com.br/robots.txt,permitido,None,HTTP 200
5,www.climatempo.com.br,https://www.climatempo.com.br/noticia/2024/09/...,https://www.climatempo.com.br/robots.txt,permitido,None,HTTP 200
6,sbtnews.sbt.com.br,https://sbtnews.sbt.com.br/noticia/brasil/cuia...,https://sbtnews.sbt.com.br/robots.txt,permitido,None,HTTP 200
7,www.poder360.com.br,https://www.poder360.com.br/poder-sustentavel/...,https://www.poder360.com.br/robots.txt,permitido,None,HTTP 200
8,www.poder360.com.br,https://www.poder360.com.br/poder-sustentavel/...,https://www.poder360.com.br/robots.txt,permitido,None,HTTP 200
9,www.poder360.com.br,https://www.poder360.com.br/poder-sustentavel/...,https://www.poder360.com.br/robots.txt,permitido,None,HTTP 200


**Leitura do resultado acima:** URLs `bloqueado` são puladas. URLs `indeterminado` também são puladas
(a menos que `ROBOTS_INDETERMINADO_PROSSEGUIR = True`). Erros 401/403 na leitura do `robots.txt`
costumam vir de proteção anti-bot (ex.: Cloudflare), não de uma política do site — nesses casos,
use a conferência manual (abaixo) ou decida conscientemente prosseguir e **registre a decisão**.


In [ ]:
def extrair_texto_pagina(html_bytes):
    '''Devolve (titulo, data_publicacao_meta, texto_do_conteudo_principal).
    Foca no <article>/<main> e remove blocos de ruído (relacionadas, comentários, compartilhar...)
    para não capturar números de OUTRAS matérias listadas na página.'''
    def _sopa():
        return BeautifulSoup(html_bytes, "lxml")

    soup = _sopa()
    titulo = soup.title.get_text(strip=True) if soup.title else None

    data_meta = None
    for attrs in ({"property": "article:published_time"}, {"name": "article:published_time"},
                  {"itemprop": "datePublished"}, {"name": "date"}, {"property": "og:updated_time"}):
        tag = soup.find("meta", attrs=attrs)
        if tag is not None and tag.get("content"):
            data_meta = tag["content"][:10]
            break
    if data_meta is None:
        t = soup.find("time", attrs={"datetime": True})
        if t is not None:
            data_meta = t["datetime"][:10]

    # Limpeza principal (NÃO remove <header>: o lead/título da matéria costuma estar nele)
    for tag in soup(["script", "style", "noscript", "nav", "footer", "aside", "form", "iframe"]):
        tag.decompose()
    ruido = re.compile(
        r"relacionad|leia-?mais|leia-?tamb|related|comment|coment|share|compartilh|"
        r"newsletter|sidebar|widget|publicidade|advert", re.I)
    for tag in soup.find_all(attrs={"class": ruido}) + soup.find_all(attrs={"id": ruido}):
        if not getattr(tag, "decomposed", False):
            tag.decompose()

    area = soup.find("article") or soup.find("main") or soup.body or soup
    texto = re.sub(r"\s+", " ", area.get_text(separator=" ")).strip()

    # Se a limpeza por classe apagou quase tudo (classe de "ruído" em um contêiner grande), refaz
    # SEM o filtro por classe/id, mas ainda sem menu/rodapé/lateral, e avisa no texto de retorno.
    if len(texto) < 300:
        soup2 = _sopa()
        for tag in soup2(["script", "style", "noscript", "nav", "footer", "aside", "form", "iframe"]):
            tag.decompose()
        estreito = re.compile(r"relacionad|related|leia-?mais|leia-?tamb", re.I)
        for tag in soup2.find_all(attrs={"class": estreito}) + soup2.find_all(attrs={"id": estreito}):
            if not getattr(tag, "decomposed", False):
                tag.decompose()
        area2 = soup2.find("article") or soup2.find("main") or soup2.body or soup2
        texto = re.sub(r"\s+", " ", area2.get_text(separator=" ")).strip()

    return titulo, data_meta, texto


# Padrões (regex) para os números dos boletins. Cada item: (regex, divisor). O grupo 1 é o número.
PADROES = {
    "temp_max_cuiaba_c": [
        (re.compile(r"(\d{2}[.,]\d)\s*°?c\s+cuiab[áa]", re.I), 1),               # formato de ranking: "41,4°C Cuiabá"
        (re.compile(r"cuiab[áa][^.]{0,150}?(\d{2}[.,]\d)\s*°?c", re.I), 1),      # formato de prosa
    ],
    "temp_max_brasil_c": [
        (re.compile(r"maiores?\s+temperaturas[^\d]{0,100}?(\d{2}[.,]\d)\s*°?c", re.I), 1),
        (re.compile(r"(\d{2}[.,]\d)\s*°?c[^.]{0,60}?maior\s+temperatura\s+do\s+(?:pa[íi]s|brasil)", re.I), 1),
    ],
    "umidade_minima_pct": [
        (re.compile(r"umidade[^.]{0,80}?abaixo\s+de\s+(\d{1,2})\s*%", re.I), 1),
        (re.compile(r"entre\s+(\d{1,2})%\s+e\s+\d{1,2}%", re.I), 1),
    ],
    "dias_consecutivos_40c": [
        (re.compile(r"(\d{1,2})\s+dias?\s+consecutivos[^.]{0,40}?40", re.I), 1),
    ],
    "focos_calor_mt": [
        (re.compile(r"mato\s+grosso[^.]{0,60}?(\d{1,3}(?:\.\d{3})*)\s+focos", re.I), 1),
        (re.compile(r"(\d{1,3}(?:\.\d{3})*)\s+focos[^.]{0,60}?mato\s+grosso", re.I), 1),
    ],
}

# Faixas plausíveis: valor fora da faixa é descartado (evita capturar números de outro contexto).
# NOTA: totais mensais/anuais de queimadas (ex.: 68.635 em agosto inteiro) caem FORA da faixa diária
# de 'focos_calor_mt' de propósito — são um agregado de granularidade diferente (ver Seção 6).
FAIXAS = {
    "temp_max_cuiaba_c": (25, 46),
    "temp_max_brasil_c": (25, 46),
    "umidade_minima_pct": (1, 40),
    "dias_consecutivos_40c": (1, 15),
    "focos_calor_mt": (1, 5000),
}


def extrair_campo(campo, texto):
    '''Primeiro casamento válido (dentro da faixa). Devolve (valor, trecho_de_evidência).'''
    lo, hi = FAIXAS[campo]
    for regex, divisor in PADROES[campo]:
        for m in regex.finditer(texto):
            v = parse_num_ptbr(m.group(1))
            if v is None:
                continue
            v = v / divisor
            if lo <= v <= hi:
                return v, texto[max(0, m.start() - 60): m.end() + 60]
    return None, None


print("Extratores prontos.")


Extratores prontos.


In [ ]:
registros_scraping = []

for item in URLS_ONDACALOR:
    url = item["url"]
    dominio = urlparse(url).netloc
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", url.rstrip("/").split("/")[-1])[:80]
    caminho_bruto = os.path.join(DIR_BRUTOS_SCRAPING, f"{item['data_ref']}_{slug}.html")

    # Respeita robots.txt (Seção 2)
    if item["robots"] == "bloqueado" or (item["robots"] == "indeterminado" and not ROBOTS_INDETERMINADO_PROSSEGUIR):
        print(f"PULADO (robots={item['robots']}) {item['data_ref']} {dominio}")
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="(não coletado)", parametros={},
            observacao=f"URL pulada: robots.txt = {item['robots']}",
        )
        continue

    try:
        resp = http_get(url, tentativas=2, timeout=20)
        resp.raise_for_status()

        # 1) PRESERVAÇÃO DO BRUTO: bytes exatamente como vieram, antes de qualquer parsing.
        sha = salvar_bruto(caminho_bruto, resp.content)
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="GET (requests + BeautifulSoup)",
            parametros={"headers": "User-Agent customizado", "timeout": 20},
            observacao=f"HTTP {resp.status_code}", arquivo=caminho_bruto, sha256=sha,
        )

        # 2) Extração (BeautifulSoup recebe bytes e detecta o encoding sozinho)
        titulo, data_meta, texto = extrair_texto_pagina(resp.content)
        registro = {
            "data_ref": item["data_ref"], "fonte": item["fonte"], "url": url, "titulo": titulo,
            "data_publicacao_meta": data_meta,
            "data_divergente": bool(data_meta and data_meta[:7] != item["data_ref"][:7]),
            "arquivo_bruto": caminho_bruto,
        }
        evidencias = {}
        for campo in PADROES:
            valor, trecho = extrair_campo(campo, texto)
            registro[campo] = valor
            if trecho:
                evidencias[campo] = trecho
        registro["evidencias"] = json.dumps(evidencias, ensure_ascii=False)
        registros_scraping.append(registro)
        print(f"OK   {item['data_ref']}  {dominio}")

    except Exception as e:
        print(f"FALHOU {item['data_ref']} {dominio}: {e}")
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="GET (requests)", parametros={}, observacao=f"ERRO: {e}",
        )

    time.sleep(max(DELAY_ENTRE_REQUISICOES, item.get("crawl_delay") or 0))

df_scraping_raw = pd.DataFrame(registros_scraping)
if len(df_scraping_raw):
    n_div = int(df_scraping_raw["data_divergente"].sum())
    if n_div:
        print(f"\n[atenção] {n_div} página(s) com data publicada em mês diferente de data_ref — confira `data_publicacao_meta`.")
df_scraping_raw


OK   2024-08-15  www.climatempo.com.br
OK   2024-08-19  ihu.unisinos.br
OK   2024-08-21  www.climatempo.com.br
OK   2024-08-29  www.climatempo.com.br
FALHOU 2024-08-30 www.poder360.com.br: 403 Client Error: Forbidden for url: https://www.poder360.com.br/brasil/brasil-tem-agosto-com-maior-numero-de-queimadas-desde-2010/
OK   2024-09-03  www.climatempo.com.br
OK   2024-09-07  sbtnews.sbt.com.br
FALHOU 2024-09-10 www.poder360.com.br: 403 Client Error: Forbidden for url: https://www.poder360.com.br/poder-sustentavel/brasil-registra-5-132-focos-de-incendio/
FALHOU 2024-09-11 www.poder360.com.br: 403 Client Error: Forbidden for url: https://www.poder360.com.br/poder-sustentavel/brasil-registra-2-909-focos-de-incendio/
FALHOU 2024-09-24 www.poder360.com.br: 403 Client Error: Forbidden for url: https://www.poder360.com.br/poder-sustentavel/brasil-registra-1-338-focos-de-incendio-sendo-57-na-amazonia/

[atenção] 4 página(s) com data publicada em mês diferente de data_ref — confira `data_publica

,data_ref,fonte,url,titulo,data_publicacao_meta,data_divergente,arquivo_bruto,temp_max_cuiaba_c,temp_max_brasil_c,umidade_minima_pct,dias_consecutivos_40c,focos_calor_mt,evidencias
0,2024-08-15,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,"Com 41°C, Cuiabá registra novo recorde de calo...",16/08/2024,True,./dados_brutos/scraping_ondacalor/2024-08-15_c...,41.0,NaN,NaN,None,None,"{""temp_max_cuiaba_c"": ""nhuma busca recente. Us..."
1,2024-08-19,IHU/ClimaInfo,https://ihu.unisinos.br/642521-clima-extremo-n...,Clima extremo: nova onda de calor no Brasil po...,None,False,./dados_brutos/scraping_ondacalor/2024-08-19_6...,NaN,NaN,10.0,None,None,"{""umidade_minima_pct"": ""crianças. Em certas re..."
2,2024-08-21,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,Cuiabá: sete dias consecutivos com mais de 40°...,22/08/2024,True,./dados_brutos/scraping_ondacalor/2024-08-21_c...,41.4,NaN,NaN,None,None,"{""temp_max_cuiaba_c"": ""/08/24 , pela medição d..."
3,2024-08-29,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,Agosto termina com mais de 40°C no Brasil | Cl...,29/08/2024,True,./dados_brutos/scraping_ondacalor/2024-08-29_a...,41.0,NaN,NaN,None,None,"{""temp_max_cuiaba_c"": ""maiores temperaturas no..."
4,2024-09-03,Climatempo,https://www.climatempo.com.br/noticia/2024/09/...,Calor acima dos 40°C e umidade abaixo de 10% n...,04/09/2024,True,./dados_brutos/scraping_ondacalor/2024-09-03_c...,42.0,NaN,10.0,None,None,"{""temp_max_cuiaba_c"": ""es temperaturas no Bras..."
5,2024-09-07,SBT News,https://sbtnews.sbt.com.br/noticia/brasil/cuia...,"Cuiabá registra 42,6°C, maior temperatura do p...",2024-09-08,False,./dados_brutos/scraping_ondacalor/2024-09-07_c...,42.6,42.6,12.0,None,None,"{""temp_max_cuiaba_c"": ""📰 Brasil Cuiabá registr..."


### Conferência manual (prioridade sobre o automático)

Como a extração automática depende de regex sobre texto jornalístico (formato não padronizado),
é normal que alguns campos venham vazios — nem toda matéria cita todos os números — ou errados.
A tabela abaixo é uma **conferência manual** das mesmas páginas de `URLS_ONDACALOR`. Na Seção 6.1
ela tem **prioridade** sobre o valor automático, e toda divergência é listada.

> ⚠️ **O grupo deve rever cada linha abaixo contra a página original antes de entregar.**


In [ ]:
dados_manuais_conferencia = [
    {"data_ref": "2024-08-15", "temp_max_cuiaba_c": 41.0, "temp_max_brasil_c": 41.0, "umidade_minima_pct": None, "dias_consecutivos_40c": 1,    "focos_calor_mt": None},
    {"data_ref": "2024-08-19", "temp_max_cuiaba_c": None, "temp_max_brasil_c": None, "umidade_minima_pct": 10,   "dias_consecutivos_40c": None, "focos_calor_mt": None},
    {"data_ref": "2024-08-21", "temp_max_cuiaba_c": 41.4, "temp_max_brasil_c": 41.4, "umidade_minima_pct": None, "dias_consecutivos_40c": 7,    "focos_calor_mt": None},
    {"data_ref": "2024-08-29", "temp_max_cuiaba_c": 41.0, "temp_max_brasil_c": 41.0, "umidade_minima_pct": None, "dias_consecutivos_40c": None, "focos_calor_mt": None},
    {"data_ref": "2024-08-30", "temp_max_cuiaba_c": None, "temp_max_brasil_c": None, "umidade_minima_pct": None, "dias_consecutivos_40c": None, "focos_calor_mt": None},
    {"data_ref": "2024-09-03", "temp_max_cuiaba_c": 42.0, "temp_max_brasil_c": 42.0, "umidade_minima_pct": 10,   "dias_consecutivos_40c": None, "focos_calor_mt": None},
    {"data_ref": "2024-09-07", "temp_max_cuiaba_c": 42.6, "temp_max_brasil_c": 42.6, "umidade_minima_pct": None, "dias_consecutivos_40c": None, "focos_calor_mt": None},
    {"data_ref": "2024-09-10", "temp_max_cuiaba_c": None, "temp_max_brasil_c": None, "umidade_minima_pct": None, "dias_consecutivos_40c": None, "focos_calor_mt": 2124},
    {"data_ref": "2024-09-11", "temp_max_cuiaba_c": None, "temp_max_brasil_c": None, "umidade_minima_pct": None, "dias_consecutivos_40c": None, "focos_calor_mt": 742},
    {"data_ref": "2024-09-24", "temp_max_cuiaba_c": None, "temp_max_brasil_c": None, "umidade_minima_pct": None, "dias_consecutivos_40c": None, "focos_calor_mt": 594},
]
df_manual = pd.DataFrame(dados_manuais_conferencia)
for c in ["temp_max_cuiaba_c", "temp_max_brasil_c", "umidade_minima_pct", "dias_consecutivos_40c", "focos_calor_mt"]:
    df_manual[c] = df_manual[c].astype(float)

caminho_manual = os.path.join(DIR_BRUTOS_SCRAPING, "conferencia_manual_boletins.csv")
df_manual.to_csv(caminho_manual, index=False)
with open(caminho_manual, "rb") as f:
    _sha_manual = sha256_bytes(f.read())
registrar_provenancia(
    fonte="Conferência manual (mesmas URLs de URLS_ONDACALOR)",
    url="ver coluna url em URLS_ONDACALOR",
    metodo="Leitura manual pelos autores do trabalho",
    parametros={}, observacao="valores digitados pelos autores; revisar contra as páginas",
    arquivo=caminho_manual, sha256=_sha_manual,
)
df_manual


,data_ref,temp_max_cuiaba_c,temp_max_brasil_c,umidade_minima_pct,dias_consecutivos_40c,focos_calor_mt
0,2024-08-15,41.0,41.0,NaN,1.0,NaN
1,2024-08-19,NaN,NaN,10.0,NaN,NaN
2,2024-08-21,41.4,41.4,NaN,7.0,NaN
3,2024-08-29,41.0,41.0,NaN,NaN,NaN
4,2024-08-30,NaN,NaN,NaN,NaN,NaN
5,2024-09-03,42.0,42.0,10.0,NaN,NaN
6,2024-09-07,42.6,42.6,NaN,NaN,NaN
7,2024-09-10,NaN,NaN,NaN,NaN,2124.0
8,2024-09-11,NaN,NaN,NaN,NaN,742.0
9,2024-09-24,NaN,NaN,NaN,NaN,594.0


## 4. Fonte 2 — API do IBGE

Duas famílias de chamadas:

1. **Localidades** — lista oficial dos municípios do Centro-Oeste (Mato Grosso, Mato Grosso do
   Sul, Goiás e Distrito Federal), obtida em uma única chamada pela **região** (código 5), sem
   precisar juntar 4 chamadas por UF. É uma dimensão de apoio para as fases futuras
   (agrupamento/geolocalização). *Não* traz população.
2. **SIDRA/Agregados** — indicadores econômicos: **IPCA mensal** (tabela 7060) e **taxa de
   desocupação** da PNAD Contínua (tabela 4099, trimestral).

**Como os parâmetros são definidos:**

- **IDs de tabela fixos** (Seção 1), conferidos em tempo de execução pelo nome e pelos metadados
  impressos abaixo — em vez de pegar o "primeiro resultado" de uma busca no catálogo, cuja ordem é arbitrária.
- **Localidade descoberta pelo nome** via `/agregados/{id}/localidades/{nivel}` (nada de código
  hardcoded): para o IPCA procura Brasília, depois Goiânia, depois Campo Grande; para a
  desocupação procura a Região Centro-Oeste e, se faltar, Mato Grosso (estado mais afetado).
- **Períodos explícitos**: lista os períodos reais da tabela (`/agregados/{id}/periodos`) e
  seleciona os que intersectam a janela `JANELA_INICIO`–`JANELA_FIM`. A frequência (mensal ou
  trimestral) é inferida dos próprios códigos de período.


In [ ]:
URL_MUNICIPIOS_CO = "https://servicodados.ibge.gov.br/api/v1/localidades/regioes/5/municipios"

resp = http_get(URL_MUNICIPIOS_CO, timeout=20)
resp.raise_for_status()
municipios_co_raw = resp.json()

caminho_bruto_municipios = os.path.join(DIR_BRUTOS_IBGE, "municipios_co_raw.json")
sha = salvar_json_bruto(caminho_bruto_municipios, municipios_co_raw)
registrar_provenancia(
    fonte="API IBGE - Localidades (Região Centro-Oeste, código 5)", url=URL_MUNICIPIOS_CO, metodo="GET",
    parametros={}, observacao=f"{len(municipios_co_raw)} municípios",
    arquivo=caminho_bruto_municipios, sha256=sha,
)

print(f"{len(municipios_co_raw)} municípios do Centro-Oeste coletados "
      "(esperado: em torno de 460-470, somando MT+MS+GO+DF).")
pd.json_normalize(municipios_co_raw)[["id", "nome"]].head()


468 municípios do Centro-Oeste coletados (esperado: em torno de 460-470, somando MT+MS+GO+DF).


,id,nome
0,5000203,Água Clara
1,5000252,Alcinópolis
2,5000609,Amambai
3,5000708,Anastácio
4,5000807,Anaurilândia


In [ ]:
BASE_API_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"
_MISSING_SIDRA = {"", "-", "..", "...", "X"}


def obter_json(url, fonte, parametros=None, caminho_bruto=None):
    '''GET + JSON + proveniência. Em caso de erro imprime o corpo da resposta (a API do IBGE
    costuma explicar o que está errado) e devolve None.'''
    try:
        resp = http_get(url, timeout=30)
    except Exception as e:
        print(f"[ERRO de rede] {fonte}: {e}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros, observacao=f"ERRO: {e}")
        return None
    if resp.status_code != 200:
        print(f"[ERRO HTTP {resp.status_code}] {fonte}\n   corpo: {resp.text[:400]}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros,
                              observacao=f"ERRO HTTP {resp.status_code}: {resp.text[:200]}")
        return None
    try:
        dados = resp.json()
    except ValueError:
        print(f"[ERRO] {fonte}: resposta não é JSON: {resp.text[:200]}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros, observacao="ERRO: resposta não-JSON")
        return None
    sha = salvar_json_bruto(caminho_bruto, dados) if caminho_bruto else None
    registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros,
                          observacao="OK", arquivo=caminho_bruto, sha256=sha)
    return dados


def niveis_da_tabela(meta):
    '''Conjunto de níveis territoriais (N1, N2, N3, N6, N7...) disponíveis na tabela.
    Aceita tanto {"Administrativo": ["N1","N3"], ...} quanto {"N3": true, ...}.'''
    nt = meta.get("nivelTerritorial", {})
    niveis = set()
    if isinstance(nt, dict):
        for k, v in nt.items():
            if isinstance(v, (list, tuple, set)):
                niveis.update(v)
            elif v and re.fullmatch(r"N\d+", str(k)):
                niveis.add(str(k))
    return niveis


def escolher_variavel(variaveis, palavras_chave):
    '''Escolhe a variável cujo nome contém alguma palavra-chave; senão, a primeira (com aviso).'''
    for v in variaveis:
        if any(p in v["nome"].lower() for p in palavras_chave):
            return v["id"]
    print(f"[aviso] nenhuma variável casou com {palavras_chave}; usando a primeira: {variaveis[0]['nome']}")
    return variaveis[0]["id"]


def montar_classificacao(meta, preferencias=("índice geral", "total", "geral")):
    '''Muitas tabelas exigem `classificacao`. Para cada classificação escolhe a categoria mais
    agregada (por nome, na ordem de `preferencias`); se nenhuma casar, usa a primeira.'''
    partes = []
    for classif in meta.get("classificacoes", []):
        cats = classif.get("categorias", [])
        if not cats:
            continue
        escolhida = None
        for pref in preferencias:
            escolhida = next((c for c in cats if pref in c["nome"].lower()), None)
            if escolhida:
                break
        escolhida = escolhida or cats[0]
        partes.append(f"{classif['id']}[{escolhida['id']}]")
    return "|".join(partes) if partes else None


def localizar_localidade(id_tabela, preferencias, niveis_disponiveis):
    '''preferencias = [(nivel, trecho_do_nome), ...] em ordem de preferência.
    Consulta /agregados/{id}/localidades/{nivel} e devolve (nivel, id, nome) do 1º casamento.'''
    cache = {}
    for nivel, trecho in preferencias:
        if nivel not in niveis_disponiveis:
            continue
        if nivel not in cache:
            cache[nivel] = obter_json(
                f"{BASE_API_AGREGADOS}/{id_tabela}/localidades/{nivel}",
                fonte=f"API IBGE - Localidades do agregado {id_tabela} ({nivel})",
                parametros={"nivel": nivel},
            ) or []
        for loc in cache[nivel]:
            if trecho in unidecode(loc["nome"]).lower():
                return nivel, str(loc["id"]), loc["nome"]
    return None


def inferir_frequencia(ids_periodos):
    '''\'trimestral\' se os códigos AAAAQQ só usam QQ = 01..04; \'mensal\' caso contrário
    (inclui trimestre móvel, em que QQ é o mês final, 01..12).'''
    sufixos = [int(p[4:]) for p in ids_periodos if len(p) == 6 and p.isdigit()]
    return "trimestral" if sufixos and max(sufixos) <= 4 else "mensal"


def periodo_para_meses(codigo, frequencia):
    '''Converte um código de período do SIDRA em lista de \'AAAA-MM\'.
    trimestral: 202304 (4º tri) -> [\'2023-10\',\'2023-11\',\'2023-12\'] | mensal: 202304 -> [\'2023-04\'].'''
    codigo = str(codigo)
    if len(codigo) != 6 or not codigo.isdigit():
        return []
    ano, suf = int(codigo[:4]), int(codigo[4:])
    if frequencia == "trimestral":
        return [f"{ano}-{m:02d}" for m in range(3 * suf - 2, 3 * suf + 1)] if 1 <= suf <= 4 else []
    return [f"{ano}-{suf:02d}"] if 1 <= suf <= 12 else []


def periodos_na_janela(id_tabela):
    '''Lista os períodos reais da tabela e devolve (ids_na_janela, frequencia).'''
    lista = obter_json(f"{BASE_API_AGREGADOS}/{id_tabela}/periodos",
                       fonte=f"API IBGE - Períodos do agregado {id_tabela}")
    if not lista:
        return [], None
    ids = [str(p["id"]) for p in lista]
    freq = inferir_frequencia(ids)
    janela = set(MESES_JANELA)
    return [i for i in ids if set(periodo_para_meses(i, freq)) & janela], freq


def montar_url_sidra(id_tabela, variavel, periodos, localidades, classificacao=None):
    url = (f"{BASE_API_AGREGADOS}/{id_tabela}/periodos/{'|'.join(periodos)}"
           f"/variaveis/{variavel}?localidades={localidades}")
    if classificacao:
        url += f"&classificacao={classificacao}"
    return url


def converter_valor_sidra(valor):
    s = str(valor).strip()
    if s in _MISSING_SIDRA:
        return None
    try:
        return float(s.replace(",", "."))
    except ValueError:
        return None


def sidra_para_df(sidra_json, nome_valor, frequencia):
    '''Achata o JSON do SIDRA (lista de variáveis -> \'resultados\' -> \'series\' -> \'serie\') em um
    DataFrame tidy [ano_mes, periodo_original, localidade, <nome_valor>]. Trimestres viram 3 meses.'''
    colunas = ["ano_mes", "periodo_original", "localidade", nome_valor]
    linhas = []
    for variavel in sidra_json:
        for resultado in variavel.get("resultados", []):
            for serie in resultado.get("series", []):
                localidade = serie.get("localidade", {}).get("nome")
                for periodo, valor in serie.get("serie", {}).items():
                    v = converter_valor_sidra(valor)
                    for ano_mes in periodo_para_meses(periodo, frequencia):
                        linhas.append({"ano_mes": ano_mes, "periodo_original": str(periodo),
                                       "localidade": localidade, nome_valor: v})
    df = pd.DataFrame(linhas, columns=colunas)
    df[nome_valor] = df[nome_valor].astype(float)
    return df.drop_duplicates(subset="ano_mes", keep="last").reset_index(drop=True)


def coletar_indicador(id_tabela, rotulo, preferencias_localidade, palavras_variavel, arquivo_bruto):
    '''Pipeline completo para um agregado: metadados -> variável -> classificação -> localidade ->
    períodos -> dados. Devolve dict com o JSON bruto e os parâmetros usados, ou None se falhar.'''
    meta = obter_json(f"{BASE_API_AGREGADOS}/{id_tabela}/metadados",
                      fonte=f"API IBGE - Metadados do agregado {id_tabela}")
    if meta is None:
        return None

    print(f"[{rotulo}] Tabela {id_tabela}: {meta.get('nome')}")
    print("   Periodicidade:", meta.get("periodicidade"))
    print("   Variáveis:")
    for v in meta.get("variaveis", []):
        print("     ", v["id"], "-", v["nome"])
    niveis = niveis_da_tabela(meta)
    print("   Níveis territoriais:", sorted(niveis))

    variavel = escolher_variavel(meta["variaveis"], palavras_variavel)
    classificacao = montar_classificacao(meta)
    achado = localizar_localidade(id_tabela, preferencias_localidade, niveis)
    if achado is None:
        print(f"[{rotulo}] Nenhuma das localidades preferidas existe nesta tabela: {preferencias_localidade}")
        return None
    nivel, loc_id, loc_nome = achado
    periodos, frequencia = periodos_na_janela(id_tabela)
    if not periodos:
        print(f"[{rotulo}] Nenhum período da tabela intersecta a janela {JANELA_INICIO}..{JANELA_FIM}.")
        return None

    url = montar_url_sidra(id_tabela, variavel, periodos, f"{nivel}[{loc_id}]", classificacao)
    print(f"   Localidade: {loc_nome} ({nivel}[{loc_id}]) | frequência: {frequencia} | "
          f"{len(periodos)} períodos | classificação: {classificacao}")
    dados = obter_json(
        url, fonte=f"API IBGE - SIDRA ({rotulo})",
        parametros={"agregado": id_tabela, "variavel": variavel, "localidades": f"{nivel}[{loc_id}]",
                    "classificacao": classificacao, "periodos": f"{periodos[0]}..{periodos[-1]}"},
        caminho_bruto=arquivo_bruto,
    )
    if dados is None:
        return None
    return {"json": dados, "frequencia": frequencia, "localidade_nome": loc_nome, "url": url,
            "tabela_nome": meta.get("nome")}


print("Funções da API prontas.")


Funções da API prontas.


In [ ]:
# (Opcional, só para conferência) O ID pinado aparece no catálogo público do SIDRA? Com que nome?
catalogo = obter_json(BASE_API_AGREGADOS, fonte="API IBGE - Catálogo de Agregados (SIDRA)")
if catalogo:
    nomes_por_id = {str(a["id"]): a["nome"] for grupo in catalogo for a in grupo.get("agregados", [])}
    for rotulo, tid in [("IPCA", ID_TABELA_IPCA), ("Desocupação", ID_TABELA_DESOCUPACAO)]:
        print(f"{rotulo}: tabela {tid} -> {nomes_por_id.get(str(tid), '!! NÃO ENCONTRADA no catálogo !!')}")
else:
    print("Catálogo indisponível; a conferência será feita pelos metadados nas próximas células.")


IPCA: tabela 7060 -> IPCA - Variação mensal, acumulada no ano, acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços (a partir de janeiro/2020)
Desocupação: tabela 4099 -> Taxas de desocupação e de subutilização da força de trabalho, na semana de referência, das pessoas de 14 anos ou mais de idade


In [ ]:
# --- IPCA: Brasília é área de coleta histórica; Goiânia/Campo Grande desde jan/2020 (tabela 7060) ---
res_ipca = coletar_indicador(
    ID_TABELA_IPCA, "IPCA",
    preferencias_localidade=[("N6", "brasilia"), ("N7", "brasilia"),
                             ("N6", "goiania"), ("N7", "goiania"),
                             ("N6", "campo grande"), ("N7", "campo grande"),
                             ("N2", "centro-oeste"), ("N1", "brasil")],   # proxies de último recurso
    palavras_variavel=["variação mensal"],
    arquivo_bruto=os.path.join(DIR_BRUTOS_IBGE, "ipca_raw.json"),
)
if res_ipca is None:
    print("\nIPCA NÃO coletado. Veja a mensagem de erro acima (corpo da resposta da API) e ajuste ID_TABELA_IPCA.")
else:
    loc_lower = unidecode(res_ipca["localidade_nome"]).lower()
    if not any(c in loc_lower for c in ("brasilia", "goiania", "campo grande")):
        print(f"\n[ATENÇÃO] Nenhuma capital do Centro-Oeste existe na tabela {ID_TABELA_IPCA} (inesperado). "
              f"Usando PROXY: '{res_ipca['localidade_nome']}'. Registre isso no dataset card (Seção 10, A.5).")


[IPCA] Tabela 7060: IPCA - Variação mensal, acumulada no ano, acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços (a partir de janeiro/2020)
   Periodicidade: {'frequencia': 'mensal', 'inicio': 202001, 'fim': 202608}
   Variáveis:
      63 - IPCA - Variação mensal
      69 - IPCA - Variação acumulada no ano
      2265 - IPCA - Variação acumulada em 12 meses
      66 - IPCA - Peso mensal
   Níveis territoriais: ['N1', 'N6', 'N7']
   Localidade: Brasília (N6[5300108]) | frequência: mensal | 16 períodos | classificação: 315[7169]


In [ ]:
# --- Taxa de desocupação (PNAD Contínua): Região Centro-Oeste; se faltar, Mato Grosso; senão Brasil ---
res_desoc = coletar_indicador(
    ID_TABELA_DESOCUPACAO, "Desocupação",
    preferencias_localidade=[("N2", "centro-oeste"), ("N3", "mato grosso"), ("N1", "brasil")],
    palavras_variavel=["taxa de desocupação", "desocupação"],
    arquivo_bruto=os.path.join(DIR_BRUTOS_IBGE, "desocupacao_raw.json"),
)
if res_desoc is None:
    print("\nDesocupação NÃO coletada. Veja a mensagem de erro acima e confira ID_TABELA_DESOCUPACAO.")
else:
    if res_desoc["frequencia"] == "trimestral":
        print("\n[nota] Série TRIMESTRAL: cada trimestre será repetido nos seus 3 meses (Seção 6.2).")
    if "centro-oeste" not in unidecode(res_desoc["localidade_nome"]).lower():
        print(f"[ATENÇÃO] Usando '{res_desoc['localidade_nome']}' no lugar da Região Centro-Oeste. Registre no dataset card.")


[Desocupação] Tabela 4099: Taxas de desocupação e de subutilização da força de trabalho, na semana de referência, das pessoas de 14 anos ou mais de idade
   Periodicidade: {'frequencia': 'trimestral', 'inicio': 201201, 'fim': 202602}
   Variáveis:
      4099 - Taxa de desocupação, na semana de referência, das pessoas de 14 anos ou mais de idade
      4103 - Coeficiente de variação - Taxa de desocupação, na semana de referência, das pessoas de 14 anos ou mais de idade
      4114 - Taxa combinada de desocupação e de subocupação por insuficiência de horas trabalhadas, na semana de referência, das pessoas de 14 anos ou mais de idade
      4115 - Coeficiente de variação - Taxa combinada de desocupação e de subocupação por insuficiência de horas trabalhadas, na semana de referência, das pessoas de 14 anos ou mais de idade
      4116 - Taxa combinada de desocupação e força de trabalho potencial, na semana de referência, das pessoas de 14 anos ou mais de idade
      4117 - Coeficiente de varia

## 5. Conferência da preservação do dado bruto (item 3.3)

In [ ]:
for pasta in [DIR_BRUTOS_SCRAPING, DIR_BRUTOS_IBGE]:
    print(f"\n{pasta}/")
    for arq in sorted(os.listdir(pasta)):
        caminho = os.path.join(pasta, arq)
        with open(caminho, "rb") as f:
            h = sha256_bytes(f.read())[:12]
        print(f"  - {arq}  ({os.path.getsize(caminho) / 1024:.1f} KB, sha256 {h}…)")



./dados_brutos/scraping_ondacalor/
  - 2024-08-15_com_41dc_cuiaba_registra_novo_recorde_de_calor_6038.html  (131.5 KB, sha256 05e91cdfd3a1…)
  - 2024-08-19_642521_clima_extremo_nova_onda_de_calor_no_brasil_pode_trazer_recordes_de_temper.html  (61.8 KB, sha256 ab1c88a0c9a5…)
  - 2024-08-21_cuiaba_sete_dias_consecutivos_com_mais_de_40dc_6108.html  (130.7 KB, sha256 3a3f0ce42488…)
  - 2024-08-29_agosto_termina_com_mais_de_40dc_no_brasil_6213.html  (131.8 KB, sha256 1bd74ab1444f…)
  - 2024-09-03_calor_acima_dos_40dc_e_umidade_abaixo_de_10_no_brasil_6269.html  (132.0 KB, sha256 7a742edb06e3…)
  - 2024-09-07_cuiaba_registra_42_6_c_maior_temperatura_do_pais_segundo_inmet.html  (197.2 KB, sha256 14e122e9d5ad…)
  - conferencia_manual_boletins.csv  (0.3 KB, sha256 a533bd774990…)

./dados_brutos/api_ibge/
  - desocupacao_raw.json  (0.7 KB, sha256 57004fcd55e6…)
  - ipca_raw.json  (1.2 KB, sha256 05c8920f617b…)
  - municipios_co_raw.json  (383.2 KB, sha256 fe6321af5ae6…)


## 6. Tratamento, limpeza e integração das fontes

### 6.1 Série da onda de calor (scraping + conferência manual)
Une o resultado automático (`df_scraping_raw`) com a conferência manual (`df_manual`) pela `data_ref`.
Para cada campo, **o valor manual tem prioridade** e a coluna `origem_<campo>` registra de onde veio
(`manual`, `automatico` ou vazio). Divergências entre os dois são listadas em `df_divergencias`
para o grupo decidir. O resultado é a base **por boletim** (`df_boletins`).

### 6.2 Indicadores econômicos (API)
O JSON do SIDRA é achatado por `sidra_para_df` (Seção 4). Séries **trimestrais** são expandidas para
os 3 meses do trimestre (o valor se repete), preservando o código original em `periodo_original`.

### 6.3 Integração (base mensal)
Chave de integração = `ano_mes`. Os boletins são **agregados por mês** (evita repetir o mesmo IPCA
em várias linhas e inflar artificialmente o n) e juntados a um **calendário completo** da janela de
análise. Assim, meses **sem boletim** continuam na base (com campos da onda de calor vazios),
mantendo o contexto econômico de antes/depois do evento. Agregações por mês: todos os campos
numéricos (`temp_max_cuiaba_c`, `temp_max_brasil_c`, `dias_consecutivos_40c`, `focos_calor_mt`) →
**máximo**; `umidade_minima_pct` → **mínimo** (menor umidade = situação mais crítica); `n_boletins`
→ contagem.

> **Rótulo `em_crise`:** `True` se `temp_max_cuiaba_c >= LIMIAR_CRISE_TEMP_C`; `False` se abaixo;
> **vazio (`<NA>`) quando não há boletim de temperatura no mês** (ausência de dado ≠ ausência de
> crise). O limiar é arbitrário. Como `em_crise` é função direta de `temp_max_cuiaba_c`, **não use
> as duas juntas** em modelos futuros (vazamento de alvo).

Nomes de município (quando usados) são normalizados com `unidecode` + `str.upper().strip()`.


In [ ]:
# --- 6.1 Série da onda de calor: manual tem prioridade; registra origem e divergências ---------
CAMPOS = ["temp_max_cuiaba_c", "temp_max_brasil_c", "umidade_minima_pct", "dias_consecutivos_40c",
          "focos_calor_mt"]
COLS_AUTO = ["data_ref", "fonte", "url", "titulo", "data_publicacao_meta", "data_divergente",
             "arquivo_bruto", "evidencias"] + CAMPOS

df_auto = df_scraping_raw.copy() if len(df_scraping_raw) else pd.DataFrame(columns=COLS_AUTO)
df_est = df_auto.merge(df_manual, on="data_ref", how="outer", suffixes=("_auto", "_manual"))

divergencias = []
for c in CAMPOS:
    manual, auto = df_est[f"{c}_manual"], df_est[f"{c}_auto"]
    df_est[c] = pd.to_numeric(manual.combine_first(auto), errors="coerce")   # manual tem prioridade
    df_est[f"origem_{c}"] = np.where(manual.notna(), "manual", np.where(auto.notna(), "automatico", None))
    dif = df_est[manual.notna() & auto.notna() & (manual != auto)]
    for _, r in dif.iterrows():
        divergencias.append({"data_ref": r["data_ref"], "fonte": r.get("fonte"), "campo": c,
                             "automatico": r[f"{c}_auto"], "manual": r[f"{c}_manual"]})

df_divergencias = pd.DataFrame(divergencias, columns=["data_ref", "fonte", "campo", "automatico", "manual"])
if len(df_divergencias):
    print(f"[atenção] {len(df_divergencias)} divergência(s) entre extração automática e conferência manual "
          "(vale o manual; confira qual está certo na página original):")
    display(df_divergencias)
else:
    print("Sem divergências entre automático e manual nos campos em que ambos existem.")

df_est["data_ref"] = pd.to_datetime(df_est["data_ref"])
df_est["ano_mes"] = df_est["data_ref"].dt.strftime("%Y-%m")

COLS_BOLETIM = (["data_ref", "ano_mes", "fonte", "url"] + CAMPOS + [f"origem_{c}" for c in CAMPOS]
                + ["data_publicacao_meta", "data_divergente"])
df_boletins = (df_est.reindex(columns=COLS_BOLETIM)
                     .sort_values(["data_ref", "fonte"]).reset_index(drop=True))
df_boletins


Sem divergências entre automático e manual nos campos em que ambos existem.


,data_ref,ano_mes,fonte,url,temp_max_cuiaba_c,temp_max_brasil_c,umidade_minima_pct,dias_consecutivos_40c,focos_calor_mt,origem_temp_max_cuiaba_c,origem_temp_max_brasil_c,origem_umidade_minima_pct,origem_dias_consecutivos_40c,origem_focos_calor_mt,data_publicacao_meta,data_divergente
0,2024-08-15,2024-08,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,41.0,41.0,NaN,1.0,NaN,manual,manual,None,manual,None,16/08/2024,True
1,2024-08-19,2024-08,IHU/ClimaInfo,https://ihu.unisinos.br/642521-clima-extremo-n...,NaN,NaN,10.0,NaN,NaN,None,None,manual,None,None,None,False
2,2024-08-21,2024-08,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,41.4,41.4,NaN,7.0,NaN,manual,manual,None,manual,None,22/08/2024,True
3,2024-08-29,2024-08,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,41.0,41.0,NaN,NaN,NaN,manual,manual,None,None,None,29/08/2024,True
4,2024-08-30,2024-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,NaN,NaN
5,2024-09-03,2024-09,Climatempo,https://www.climatempo.com.br/noticia/2024/09/...,42.0,42.0,10.0,NaN,NaN,manual,manual,manual,None,None,04/09/2024,True
6,2024-09-07,2024-09,SBT News,https://sbtnews.sbt.com.br/noticia/brasil/cuia...,42.6,42.6,12.0,NaN,NaN,manual,manual,automatico,None,None,2024-09-08,False
7,2024-09-10,2024-09,NaN,NaN,NaN,NaN,NaN,NaN,2124.0,None,None,None,None,manual,NaN,NaN
8,2024-09-11,2024-09,NaN,NaN,NaN,NaN,NaN,NaN,742.0,None,None,None,None,manual,NaN,NaN
9,2024-09-24,2024-09,NaN,NaN,NaN,NaN,NaN,NaN,594.0,None,None,None,None,manual,NaN,NaN


In [ ]:
# --- 6.2 Indicadores do IBGE em DataFrames tidy: ano_mes + valor -------------------------------
def _df_indicador(res, nome_valor, prefixo):
    colunas = ["ano_mes", nome_valor, f"{prefixo}_localidade", f"{prefixo}_periodo_original"]
    if res is None:
        print(f"[aviso] {nome_valor} não foi coletado na Seção 4 — seguindo com DataFrame vazio "
              "(a coluna ficará toda vazia na base).")
        return pd.DataFrame(columns=colunas)
    df = sidra_para_df(res["json"], nome_valor, res["frequencia"])
    return df.rename(columns={"localidade": f"{prefixo}_localidade",
                              "periodo_original": f"{prefixo}_periodo_original"})[colunas]


df_ipca = _df_indicador(res_ipca if "res_ipca" in globals() else None, "ipca_variacao_mensal", "ipca")
df_desoc = _df_indicador(res_desoc if "res_desoc" in globals() else None, "taxa_desocupacao", "desocupacao")

print("IPCA:", df_ipca.shape, "| Desocupação:", df_desoc.shape)
print("\nIPCA (amostra):")
display(df_ipca.head())
print("\nDesocupação (amostra — confira se cada trimestre aparece repetido nos 3 meses):")
display(df_desoc.head(6))


IPCA: (16, 4) | Desocupação: (18, 4)

IPCA (amostra):


,ano_mes,ipca_variacao_mensal,ipca_localidade,ipca_periodo_original
0,2023-09,0.29,Brasília (DF),202309
1,2023-10,0.62,Brasília (DF),202310
2,2023-11,0.40,Brasília (DF),202311
3,2023-12,0.78,Brasília (DF),202312
4,2024-01,-0.36,Brasília (DF),202401



Desocupação (amostra — confira se cada trimestre aparece repetido nos 3 meses):


,ano_mes,taxa_desocupacao,desocupacao_localidade,desocupacao_periodo_original
0,2023-07,5.4,Centro-Oeste,202303
1,2023-08,5.4,Centro-Oeste,202303
2,2023-09,5.4,Centro-Oeste,202303
3,2023-10,5.7,Centro-Oeste,202304
4,2023-11,5.7,Centro-Oeste,202304
5,2023-12,5.7,Centro-Oeste,202304


In [ ]:
# --- 6.3 Integração final (base mensal): calendário da janela + onda de calor + IPCA + desocupação ---
agg_boletins = (df_boletins.groupby("ano_mes")
                .agg(n_boletins=("data_ref", "size"),
                     temp_max_cuiaba_c=("temp_max_cuiaba_c", "max"),
                     temp_max_brasil_c=("temp_max_brasil_c", "max"),
                     umidade_minima_pct=("umidade_minima_pct", "min"),
                     dias_consecutivos_40c=("dias_consecutivos_40c", "max"),
                     focos_calor_mt=("focos_calor_mt", "max"))
                .reset_index())

fora_da_janela = sorted(set(agg_boletins["ano_mes"]) - set(MESES_JANELA))
if fora_da_janela:
    print(f"[aviso] boletins em meses fora da janela (serão descartados da base mensal): {fora_da_janela}")

calendario = pd.DataFrame({"ano_mes": MESES_JANELA})
base_mensal = (calendario
               .merge(agg_boletins, on="ano_mes", how="left")
               .merge(df_ipca, on="ano_mes", how="left")
               .merge(df_desoc, on="ano_mes", how="left"))
base_mensal["n_boletins"] = base_mensal["n_boletins"].fillna(0).astype(int)

# Rótulo com dado ausente preservado (NÃO transformar NaN em "sem crise")
_t = base_mensal["temp_max_cuiaba_c"]
base_mensal["em_crise"] = (_t >= LIMIAR_CRISE_TEMP_C).astype("boolean").mask(_t.isna())

# Dimensão de municípios do Centro-Oeste (código, nome, nome normalizado) — apoio para fases futuras.
# Ainda NÃO entra no merge: nenhuma das fontes atuais tem dado em nível de município.
df_municipios_co = pd.json_normalize(municipios_co_raw)[["id", "nome"]].rename(
    columns={"id": "municipio_id", "nome": "municipio_nome"})
df_municipios_co["municipio_nome_normalizado"] = df_municipios_co["municipio_nome"].apply(
    lambda s: unidecode(s).upper().strip())

base_mensal


,ano_mes,n_boletins,temp_max_cuiaba_c,temp_max_brasil_c,umidade_minima_pct,dias_consecutivos_40c,focos_calor_mt,ipca_variacao_mensal,ipca_localidade,ipca_periodo_original,taxa_desocupacao,desocupacao_localidade,desocupacao_periodo_original,em_crise
0,2023-09,0,NaN,NaN,NaN,NaN,NaN,0.29,Brasília (DF),202309,5.4,Centro-Oeste,202303,<NA>
1,2023-10,0,NaN,NaN,NaN,NaN,NaN,0.62,Brasília (DF),202310,5.7,Centro-Oeste,202304,<NA>
2,2023-11,0,NaN,NaN,NaN,NaN,NaN,0.40,Brasília (DF),202311,5.7,Centro-Oeste,202304,<NA>
3,2023-12,0,NaN,NaN,NaN,NaN,NaN,0.78,Brasília (DF),202312,5.7,Centro-Oeste,202304,<NA>
4,2024-01,0,NaN,NaN,NaN,NaN,NaN,-0.36,Brasília (DF),202401,6.0,Centro-Oeste,202401,<NA>
5,2024-02,0,NaN,NaN,NaN,NaN,NaN,0.75,Brasília (DF),202402,6.0,Centro-Oeste,202401,<NA>
6,2024-03,0,NaN,NaN,NaN,NaN,NaN,0.21,Brasília (DF),202403,6.0,Centro-Oeste,202401,<NA>
7,2024-04,0,NaN,NaN,NaN,NaN,NaN,0.55,Brasília (DF),202404,5.4,Centro-Oeste,202402,<NA>
8,2024-05,0,NaN,NaN,NaN,NaN,NaN,0.34,Brasília (DF),202405,5.4,Centro-Oeste,202402,<NA>
9,2024-06,0,NaN,NaN,NaN,NaN,NaN,0.34,Brasília (DF),202406,5.4,Centro-Oeste,202402,<NA>


In [ ]:
# --- Verificações de qualidade (rode e leia antes de salvar) --------------------------------
print("Dimensões:", base_mensal.shape, "| meses com boletim:", int((base_mensal["n_boletins"] > 0).sum()))
print("\n% de valores vazios por coluna:")
print((base_mensal.isna().mean() * 100).round(1).to_string())

alertas = []
if base_mensal["ipca_variacao_mensal"].isna().all():
    alertas.append("IPCA todo vazio: coleta falhou ou os períodos não intersectam a janela.")
if base_mensal["taxa_desocupacao"].isna().all():
    alertas.append("Desocupação toda vazia: coleta falhou ou os períodos não intersectam a janela.")
if base_mensal["n_boletins"].sum() == 0:
    alertas.append("Nenhum boletim na base mensal: scraping falhou/pulado e a conferência manual está fora da janela?")
if base_mensal["ano_mes"].duplicated().any():
    alertas.append("Há meses duplicados na base mensal (chave ano_mes deveria ser única).")
faltando = [m for m in ["2024-08", "2024-09"] if m not in set(agg_boletins["ano_mes"])]
if faltando:
    alertas.append(f"Sem boletim nos meses centrais da onda de calor de 2024: {faltando}.")

print("\nALERTAS:" if alertas else "\nSem alertas automáticos.")
for a in alertas:
    print(" -", a)


Dimensões: (16, 14) | meses com boletim: 2

% de valores vazios por coluna:
ano_mes                          0.0
n_boletins                       0.0
temp_max_cuiaba_c               87.5
temp_max_brasil_c               87.5
umidade_minima_pct              87.5
dias_consecutivos_40c           93.8
focos_calor_mt                  93.8
ipca_variacao_mensal             0.0
ipca_localidade                  0.0
ipca_periodo_original            0.0
taxa_desocupacao                 0.0
desocupacao_localidade           0.0
desocupacao_periodo_original     0.0
em_crise                        87.5

Sem alertas automáticos.


> **Nota de limitação (registrar no *dataset card*, item A.5):** a série de scraping mistura
> duas granularidades de fonte (temperatura via Climatempo/SBT e focos de calor via Poder360), em
> datas nem sempre coincidentes, então vários meses têm só um dos dois tipos de dado. Para as fases
> futuras (EDA, séries temporais), o grupo deve ampliar a coleta — idealmente com uma fonte
> estruturada (ex.: BDMEP do INMET para temperatura diária de Cuiabá, e o BDQueimadas do Inpe via
> API para focos diários) em vez de matérias de imprensa esparsas. Isso está documentado como
> *lacuna conhecida*.

## 7. Base tratada — salvar em CSV e Parquet


In [ ]:
caminho_csv = os.path.join(DIR_TRATADOS, "base_calor_indicadores_co.csv")
caminho_parquet = os.path.join(DIR_TRATADOS, "base_calor_indicadores_co.parquet")
caminho_boletins = os.path.join(DIR_TRATADOS, "base_boletins_ondacalor.csv")
caminho_municipios_tratado = os.path.join(DIR_TRATADOS, "municipios_centro_oeste.csv")

base_mensal.to_csv(caminho_csv, index=False)
try:
    base_mensal.to_parquet(caminho_parquet, index=False)
except Exception as e:
    print("Não foi possível salvar em Parquet (instale 'pyarrow'):", e)
df_boletins.to_csv(caminho_boletins, index=False)
df_municipios_co.to_csv(caminho_municipios_tratado, index=False)

print("Salvos em:")
for c in [caminho_csv, caminho_parquet, caminho_boletins, caminho_municipios_tratado]:
    print(" ", c)
print("\nBase mensal (principal):", base_mensal.shape, "| Base por boletim:", df_boletins.shape)


Salvos em:
  ./dados_tratados/base_calor_indicadores_co.csv
  ./dados_tratados/base_calor_indicadores_co.parquet
  ./dados_tratados/base_boletins_ondacalor.csv
  ./dados_tratados/municipios_centro_oeste.csv

Base mensal (principal): (16, 14) | Base por boletim: (10, 16)


## 8. Registro de proveniência (item 3.4) — salvar log completo e empacotar

In [ ]:
df_provenance = pd.DataFrame(provenance_log)
caminho_provenance = os.path.join(BASE_DIR, "registro_proveniencia.csv")
df_provenance.to_csv(caminho_provenance, index=False)
print(f"{len(df_provenance)} eventos registrados em {caminho_provenance}")
df_provenance


25 eventos registrados em ./registro_proveniencia.csv


,fonte,url,data_hora_coleta_utc,metodo,parametros,observacao,arquivo_bruto,sha256
0,robots.txt de www.climatempo.com.br,https://www.climatempo.com.br/robots.txt,2026-09-19T19:04:38+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
1,robots.txt de ihu.unisinos.br,https://ihu.unisinos.br/robots.txt,2026-09-19T19:04:39+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
2,robots.txt de www.poder360.com.br,https://www.poder360.com.br/robots.txt,2026-09-19T19:04:39+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
3,robots.txt de sbtnews.sbt.com.br,https://sbtnews.sbt.com.br/robots.txt,2026-09-19T19:04:39+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
4,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,2026-09-19T19:04:40+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_ondacalor/2024-08-15_c...,05e91cdfd3a1671b73d92b811b2fc71af382de9b958da7...
5,IHU/ClimaInfo,https://ihu.unisinos.br/642521-clima-extremo-n...,2026-09-19T19:04:43+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_ondacalor/2024-08-19_6...,ab1c88a0c9a547bf5fba1db90c25a43eb39fb80a21de83...
6,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,2026-09-19T19:04:46+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_ondacalor/2024-08-21_c...,3a3f0ce4248832a09cad424b82c2cf8b89f1e0eab5f3ee...
7,Climatempo,https://www.climatempo.com.br/noticia/2024/08/...,2026-09-19T19:04:49+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_ondacalor/2024-08-29_a...,1bd74ab1444fdd855f63445d93676385efd74fa31bce58...
8,Poder360,https://www.poder360.com.br/brasil/brasil-tem-...,2026-09-19T19:04:51+00:00,GET (requests),{},ERRO: 403 Client Error: Forbidden for url: htt...,None,None
9,Climatempo,https://www.climatempo.com.br/noticia/2024/09/...,2026-09-19T19:04:53+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_ondacalor/2024-09-03_c...,7a742edb06e3514023e5a2fd404fc7ccf10aca5f000e5e...


In [ ]:
# Empacota a entrega. O Colab apaga o disco ao desconectar: baixe o zip (ou use MONTAR_DRIVE=True).
# HTML bruto de matérias é conteúdo protegido por direitos autorais: por padrão fica FORA do zip.
caminho_zip = os.path.join(BASE_DIR, "entrega_trabalho1.zip")
with zipfile.ZipFile(caminho_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(caminho_provenance, "registro_proveniencia.csv")
    for pasta in [DIR_TRATADOS, DIR_BRUTOS]:
        for raiz, _, arquivos in os.walk(pasta):
            for a in arquivos:
                if a.endswith(".html") and not INCLUIR_HTML_BRUTO_NO_ZIP:
                    continue
                caminho = os.path.join(raiz, a)
                z.write(caminho, os.path.relpath(caminho, BASE_DIR))
print("Zip criado:", caminho_zip, f"({os.path.getsize(caminho_zip) / 1024:.0f} KB)")

try:
    from google.colab import files
    files.download(caminho_zip)
except ImportError:
    print("(fora do Colab: pegue o zip no caminho acima)")


Zip criado: ./entrega_trabalho1.zip (28 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Postura ética e legal (item 3.5)

- **robots.txt:** verificado programaticamente para **cada URL** (Seção 2). O resultado desta
  execução está em `df_robots`; o resumo é impresso na célula abaixo — **copie-o para o dataset
  card (A.6)**. URLs `bloqueado`/`indeterminado` foram puladas (salvo decisão explícita do grupo).
- **Licença / termos de uso de cada fonte:**
  - *Portais de notícia* (Climatempo, IHU/Unisinos, Poder360, SBT News): conteúdo com direitos
    autorais jornalísticos. **Não reproduzimos o texto das matérias** na base tratada — extraímos
    apenas **fatos numéricos** (temperatura, umidade, focos de calor), com o link da fonte para
    atribuição. O HTML bruto é guardado apenas para auditoria/reprodutibilidade e **não deve ser
    redistribuído** (repositório público, entrega com HTMLs): por isso fica fora do zip por padrão.
    Os trechos curtos em `evidencias` (`df_scraping_raw`) servem à conferência interna — se a base
    for publicada, remova essa coluna.
  - *API do IBGE* (Localidades e SIDRA): dados públicos abertos, disponibilizados pelo próprio órgão
    federal para reuso, inclusive acadêmico ([Portal de Serviços do IBGE](https://servicodados.ibge.gov.br/api/docs)).
- **Dados pessoais / LGPD:** a base não contém dados pessoais identificáveis — os números
  (temperatura, umidade, focos de calor) são medições ambientais agregadas, sem identificação
  individual. Não se aplica minimização/anonimização adicional.
- **Não sobrecarregar servidores:** delay de `DELAY_ENTRE_REQUISICOES` (ou o `Crawl-delay` do site,
  se maior) entre requisições de scraping, sem paralelismo; retries limitados (2–3 tentativas com
  espera crescente). O total de requisições fica no log de proveniência.


In [ ]:
# Resumo para copiar no dataset card (A.6) — gerado a partir do que realmente aconteceu nesta execução
print("robots.txt — resultado por URL:")
print(df_robots["resultado"].value_counts().to_string())
puladas = df_robots[df_robots["resultado"] != "permitido"]
if len(puladas):
    print("\nURLs NÃO raspadas (bloqueadas/indeterminadas):")
    for _, r in puladas.iterrows():
        print(f" - {r['url']}  [{r['resultado']}; {r['detalhe']}]")
print("\nRequisições registradas no log:", len(provenance_log))


robots.txt — resultado por URL:
resultado
permitido    10

Requisições registradas no log: 25


## 10. Dataset Card (Apêndice A)

### A.1 Identificação
- **Nome da base:** Onda de Calor Centro-Oeste ago/2024 × Indicadores Econômicos

### A.2 Fontes e proveniência
- **Fonte 1 — nome e URLs:** boletins/matérias sobre a onda de calor de agosto/2024 (ver
  `URLS_ONDACALOR`: Climatempo, IHU/Unisinos, Poder360 e SBT News).
- **Fonte 1 — método:** Web scraping (`requests` + `BeautifulSoup`), regex com validação de faixa
  + conferência manual (que tem prioridade; origem de cada valor em `origem_*`).
- **Fonte 1 — licença/termos:** conteúdo jornalístico; usamos apenas fatos numéricos, com atribuição de URL.
- **Fonte 2 — nome e URL:** API do IBGE — Localidades (`servicodados.ibge.gov.br/api/v1/localidades`)
  e SIDRA/Agregados (`servicodados.ibge.gov.br/api/v3/agregados`): tabela 7060 (IPCA) e tabela 4099
  (PNAD Contínua trimestral — taxa de desocupação).
- **Fonte 2 — método:** API REST (JSON).
- **Fonte 2 — licença/termos:** dados públicos abertos do IBGE.
- **Chave de integração:** `ano_mes` (data do boletim arredondada para o mês, casada com a
  granularidade mensal do IPCA e com os meses de cada trimestre da PNAD).

### A.3 Dicionário de variáveis (`base_calor_indicadores_co.csv` — base mensal)

| Variável | Tipo | Descrição | Unidade |
|---|---|---|---|
| ano_mes | categórica | Mês de referência (chave; um registro por mês da janela) | AAAA-MM |
| n_boletins | numérica discreta | Nº de boletins/matérias coletados no mês | contagem |
| temp_max_cuiaba_c | numérica contínua | Temperatura máxima registrada em Cuiabá (máx. do mês) | °C |
| temp_max_brasil_c | numérica contínua | Maior temperatura registrada no Brasil (máx. do mês) | °C |
| umidade_minima_pct | numérica contínua | Umidade relativa mínima do ar mencionada (mín. do mês) | % |
| dias_consecutivos_40c | numérica discreta | Dias consecutivos com temperatura ≥ 40°C em Cuiabá (máx. do mês) | contagem |
| focos_calor_mt | numérica discreta | Focos de calor em Mato Grosso em 24h (Inpe/BDQueimadas; máx. do mês) | contagem |
| ipca_variacao_mensal | numérica contínua | Variação mensal do IPCA na localidade de `ipca_localidade` | % |
| ipca_localidade | categórica | Localidade efetivamente usada no IPCA (Brasília, Goiânia ou Campo Grande) | - |
| ipca_periodo_original | categórica | Código do período no SIDRA | AAAAMM |
| taxa_desocupacao | numérica contínua | Taxa de desocupação (PNAD Contínua; **trimestral repetida nos 3 meses**) | % |
| desocupacao_localidade | categórica | Localidade da desocupação (Centro-Oeste ou fallback) | - |
| desocupacao_periodo_original | categórica | Trimestre original no SIDRA (AAAA0T) | AAAA01–AAAA04 |
| em_crise | booleana anulável | `True` se `temp_max_cuiaba_c >= 40`; `<NA>` se não há boletim de temperatura no mês | - |

`base_boletins_ondacalor.csv` (granularidade de boletim): `data_ref`, `ano_mes`, `fonte`, `url`, os 5
campos numéricos, `origem_<campo>` (manual/automatico), `data_publicacao_meta`, `data_divergente`.
`municipios_centro_oeste.csv`: `municipio_id`, `municipio_nome`, `municipio_nome_normalizado`.

### A.4 Volume e granularidade
- **Nº de linhas / colunas:** ver a célula de resumo logo abaixo (preencher aqui após rodar).
- **O que representa uma linha:** um **mês** da janela de análise, com o resumo dos boletins do mês
  (quando houver) e os indicadores econômicos do mês.
- **Cobertura:** Centro-Oeste (MT, MS, GO, DF) para a onda de calor (com foco em Cuiabá/MT, o
  epicentro); IPCA de capital do Centro-Oeste e PNAD da Região Centro-Oeste (ver colunas
  `*_localidade`); janela `2023-09` a `2024-12` (boletins concentrados em ago–set/2024).

### A.5 Limitações e decisões
- **Dados descartados:** URLs puladas por `robots.txt` (ver Seção 9); valores extraídos fora da
  faixa plausível são descartados (ex.: totais mensais de queimadas não entram em `focos_calor_mt`,
  que é uma contagem diária); boletins fora da janela não entram na base mensal.
- **Lacunas conhecidas:** série de scraping mistura granularidades de fonte (temperatura vs. focos
  de calor, em dias diferentes); poucos meses com boletim; conferência manual a ser revalidada
  pelo grupo.
- **Decisões de limpeza relevantes:** manual > automático (com `origem_*`); agregação mensal por
  **máximo** (temperatura, dias consecutivos, focos) e por **mínimo** (umidade, onde menor = pior);
  `em_crise` com limiar arbitrário (40°C em Cuiabá) e sem transformar ausência em `False`;
  nomes de município normalizados com `unidecode`.

### A.6 Considerações éticas
- **Contém dados pessoais?** Não.
- **Restrições de uso/redistribuição:** ver Seção 9 (não redistribuir o HTML/texto das matérias;
  dados do IBGE são abertos).
- **robots.txt verificado?** Sim, por URL — copiar o resumo da célula da Seção 9.


In [ ]:
# Resumo para preencher o A.4 do dataset card
print("Base mensal:", base_mensal.shape[0], "linhas x", base_mensal.shape[1], "colunas")
print("Base por boletim:", df_boletins.shape[0], "linhas x", df_boletins.shape[1], "colunas")
print("Período:", base_mensal["ano_mes"].min(), "a", base_mensal["ano_mes"].max())
print("Meses com boletim:", ", ".join(base_mensal.loc[base_mensal["n_boletins"] > 0, "ano_mes"]))


Base mensal: 16 linhas x 14 colunas
Base por boletim: 10 linhas x 16 colunas
Período: 2023-09 a 2024-12
Meses com boletim: 2024-08, 2024-09


Limitações Conhecidas

A natureza das fontes e a metodologia de agregação apresentam limitações inerentes que devem ser consideradas em análises preditivas ou estatísticas futuras.

1. Limitações Metodológicas

Viés de Amostragem na Agregação Climática: As variáveis meteorológicas (temperatura máxima, umidade mínima e focos de calor) foram extraídas de uma amostra reduzida de boletins noticiosos mensais (tipicamente 2 a 3 dias por mês). Aplicar funções de agregação (como max ou min) a esta amostra esparsa pode não representar a realidade climática consolidada do mês, uma vez que eventos meteorológicos isolados podem distorcer o perfil mensal.

Fragilidade na Extração de Textos Não Estruturados: A captura de dados numéricos em textos jornalísticos via expressões regulares (RegEx), apesar de possuir validações, é suscetível a ruídos semânticos. Existe o risco de o algoritmo capturar a "sensação térmica", previsões climáticas para dias subsequentes ou dados de localidades vizinhas citadas no corpo da notícia, em vez do valor efetivamente registrado na cidade-alvo.

Distorção de Granularidade Temporal (PNAD): A base de dados integra fontes mensais com dados econômicos trimestrais (PNAD Contínua). A expansão artificial do dado trimestral para os três meses correspondentes cria patamares estáticos (ausência de variação intra-trimestral). Essa "linha reta" artificial mascara a dinâmica real do mercado de trabalho mês a mês, o que pode atenuar a sensibilidade de modelos de correlação temporal.